# India Climate & Energy Atlas
## A reproducible, real-data exploration of development and the energy transition

**Data source:** World Bank Indicators API.  
**Scope:** India plus Bangladesh, China, Indonesia, Sri Lanka and Vietnam.  
**Goal:** describe long-run changes in GDP per capita, electricity access, renewable energy consumption and per-capita CO₂ emissions, while documenting data gaps and limits.

> This is an observational analysis. It does not identify causal effects. No synthetic observations are generated or used.

## 1. Reproduce the data download

Run from the project root. The API call requires an internet connection. If you already fetched the data, this cell uses the existing downloaded panel.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "processed" / "world_bank_panel.csv"
if not DATA_PATH.exists():
    subprocess.run([sys.executable, "scripts/fetch_data.py", "--start", "2000", "--end", "2023"], check=True)
print("Dataset:", DATA_PATH)

## 2. Load, audit and record coverage

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.atlas.analysis import coverage_table, trend_summary, correlation_within_country, GDP, CO2, RENEWABLES, ELECTRICITY

df = pd.read_csv(DATA_PATH)
print(f"{len(df):,} country-year rows | {df.country_code.nunique()} countries | years {int(df.year.min())}–{int(df.year.max())}")
display(df.head())
coverage = coverage_table(df)
display(coverage)
print("Never interpret a missing value as zero.")

## 3. India: observed trends

Each chart uses the observed values for that indicator. Different indicators may have different final observation years.

In [ ]:
india = df[df.country_code == "IND"].sort_values("year")
core_metrics = [GDP, CO2, ELECTRICITY, RENEWABLES]
for metric in core_metrics:
    if metric in india.columns and india[metric].notna().sum() > 1:
        subset = india[["year", metric]].dropna()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(subset["year"], subset[metric], marker="o", linewidth=2)
        ax.set_title(metric)
        ax.set_xlabel("Year")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=.25)
        fig.tight_layout()
        plt.show()
    else:
        print("Insufficient observed data:", metric)

## 4. Endpoint changes and reporting windows

Endpoint changes compare the first and last non-missing observations; they do not imply a smooth annual change.

In [ ]:
trend_rows = []
for metric in core_metrics:
    if metric in df.columns:
        trend_rows.append(trend_summary(df, metric))
display(pd.DataFrame(trend_rows))

## 5. GDP per capita and CO₂: exploratory association

The calculation uses years where both values are observed. Time-trending series can show correlation even without a direct relationship, so this is not causal evidence.

In [ ]:
if GDP in df.columns and CO2 in df.columns:
    paired = india[["year", GDP, CO2]].dropna().sort_values("year")
    print("Complete annual pairs:", len(paired))
    if len(paired) >= 3:
        print("Pearson r:", round(paired[GDP].corr(paired[CO2]), 3))
        print("Spearman rho:", round(paired[GDP].corr(paired[CO2], method="spearman"), 3))
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(paired[GDP], paired[CO2])
        for _, row in paired.iterrows():
            ax.annotate(str(int(row["year"])), (row[GDP], row[CO2]), fontsize=7, alpha=.65)
        ax.set_xlabel(GDP)
        ax.set_ylabel(CO2)
        ax.set_title("India: GDP per capita vs CO₂ emissions per capita")
        ax.grid(True, alpha=.2)
        fig.tight_layout()
        plt.show()
    else:
        print("Not enough complete pairs for correlation.")
else:
    print("Required indicator missing from downloaded panel.")

## 6. Country comparison on a consistent period

Report country differences only with coverage in view. A shared year window is preferable when making direct comparisons.

In [ ]:
if GDP in df.columns and CO2 in df.columns:
    countries = df[df.country_code.isin(["IND","BGD","CHN","IDN","LKA","VNM"])]
    paired_panel = countries.dropna(subset=[GDP, CO2]).copy()
    if not paired_panel.empty:
        display(paired_panel.groupby(["country_code","country"]).agg(
            first_year=("year","min"), last_year=("year","max"), n_complete_years=("year","size"),
            mean_gdp_per_capita=(GDP,"mean"), mean_co2_per_capita=(CO2,"mean")
        ).reset_index())
    else:
        print("No complete paired observations in the selected panel.")

## 7. Save reproducible report outputs

In [ ]:
reports = ROOT / "reports"
reports.mkdir(exist_ok=True)
coverage.to_csv(reports / "coverage.csv", index=False)
summary = {
    "rows": int(len(df)),
    "countries": sorted(df.country_code.dropna().unique().tolist()),
    "year_min": int(df.year.min()),
    "year_max": int(df.year.max()),
    "coverage": coverage.to_dict(orient="records"),
    "india_trends": trend_rows,
    "warning": "Descriptive analysis only; no causal claims."
}
(reports / "notebook_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("Saved reports/coverage.csv and reports/notebook_summary.json")

## 8. Research-quality checklist before publishing

- [ ] Cite the World Bank indicator pages and retrieval date.
- [ ] State the exact year range and number of complete observations for every analysis.
- [ ] Explain missingness and revisions.
- [ ] Compare countries over overlapping years where possible.
- [ ] Do not call correlation causation.
- [ ] Do not claim “decoupling” without a precise definition and a suitable robustness analysis.
- [ ] Re-run all cells from a clean kernel before publishing.
- [ ] Include the limitations in the Kaggle notebook description.